# Elliott and Sparrow 2012 Beam Launcher

Use this notebook to reproduce the paper-style flexible-beam waveforms and, when a Digifly Phase 2 run is available, display the recorded soma voltage traces for the GF/TTMn escape cells. No terminal commands are required.

## Setup

This cell finds the app whether the notebook is opened from the standalone project or from `Digifly Public/Phase 2/apps/Elliott_Sparrow_2012_Beam`. It also reuses Phase 2's existing `records.csv` voltage-trace conventions.

In [ ]:
from pathlib import Path
import json
import re
import sys
from typing import Iterable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


ESCAPE_CELL_IDS = [10000, 10002, 10068, 10110]
CELL_LABELS = {
    10000: "GF",
    10002: "GF",
    10068: "TTMn",
    10110: "TTMn",
    11446: "PSI",
    11654: "PSI",
}


def find_app_root() -> Path:
    marker = Path("tools") / "beam_waveform_model.py"
    starts = [Path.cwd(), *Path.cwd().parents]
    known = [
        Path("/Users/juanlopez2016/Desktop/Digifly Public/Phase 2/apps/Elliott_Sparrow_2012_Beam"),
        Path("/Users/juanlopez2016/Desktop/Elliott_Sparrow_2012_Beam"),
    ]
    for base in [*starts, *known]:
        try:
            base = base.expanduser().resolve()
        except Exception:
            continue
        if (base / marker).exists():
            return base
        nested = base / "Phase 2" / "apps" / "Elliott_Sparrow_2012_Beam"
        if (nested / marker).exists():
            return nested.resolve()
    raise RuntimeError("Could not find tools/beam_waveform_model.py. Open this notebook from the Elliott_Sparrow_2012_Beam app folder.")


def find_phase2_root(app_root: Path) -> Path | None:
    if app_root.parts[-3:] == ("Phase 2", "apps", "Elliott_Sparrow_2012_Beam"):
        return app_root.parents[1]
    for parent in [app_root, *app_root.parents]:
        if parent.name == "Phase 2" and (parent / "digifly").exists():
            return parent
    known = Path("/Users/juanlopez2016/Desktop/Digifly Public/Phase 2")
    return known.resolve() if (known / "digifly").exists() else None


APP_ROOT = find_app_root()
PHASE2_ROOT = find_phase2_root(APP_ROOT)
DIGIFLY_ROOT = PHASE2_ROOT.parent if PHASE2_ROOT is not None else None
TOOLS_DIR = APP_ROOT / "tools"

for candidate in [TOOLS_DIR, PHASE2_ROOT]:
    if candidate is not None and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from beam_waveform_model import BeamParams, generate_condition, write_outputs  # noqa: E402

# Phase 2 imports are intentionally lazy. Some NEURON/display stacks can fail
# during import in headless kernels, but the beam-only workflow should still open.
PHASE2_IMPORT_ERROR = None

DEFAULT_OUTPUT_ROOT = (
    PHASE2_ROOT / "workbench_runs" / "elliott_sparrow_beam"
    if PHASE2_ROOT is not None
    else APP_ROOT / "outputs"
)

DEFAULT_PHASE2_RUNS_ROOT = (
    DIGIFLY_ROOT / "Phase 1" / "manc_v1.2.1" / "export_swc" / "hemi_runs"
    if DIGIFLY_ROOT is not None
    else APP_ROOT / "runs"
)

CONDITIONS = [
    "wildtype_one_leg",
    "wildtype_jump",
    "cs_jump",
    "shakB2_one_leg",
    "shakB2_six_leg",
    "amph26_jump",
    "walking",
    "adhesion_grip",
    "flight_downdraft",
    "larval_wildtype",
    "larval_parkin25",
]

DEFAULT_DURATIONS_MS = {
    "walking": 1200.0,
    "adhesion_grip": 600.0,
    "flight_downdraft": 1200.0,
    "larval_wildtype": 20000.0,
    "larval_parkin25": 20000.0,
}

DEFAULT_DT_MS = {
    "larval_wildtype": 2.0,
    "larval_parkin25": 2.0,
}

display(Markdown(
    f"**App root:** `{APP_ROOT}`  \n"
    f"**Digifly Phase 2 root:** `{PHASE2_ROOT if PHASE2_ROOT else 'not found'}`  \n"
    f"**Default beam output root:** `{DEFAULT_OUTPUT_ROOT}`  \n"
    f"**Default Phase 2 run root:** `{DEFAULT_PHASE2_RUNS_ROOT}`"
))

if PHASE2_IMPORT_ERROR is not None:
    display(Markdown(f"Phase 2 simulation launch is unavailable in this kernel: `{PHASE2_IMPORT_ERROR}`"))

## Shared Plot Helpers

The voltage helper follows Phase 2's existing convention: a run folder contains `records.csv`, with a time column such as `t_ms` and soma-voltage columns like `10000_soma_v`.

In [ ]:
def default_duration_ms(condition: str) -> float:
    return DEFAULT_DURATIONS_MS.get(condition, 80.0)


def default_dt_ms(condition: str) -> float:
    return DEFAULT_DT_MS.get(condition, 0.025)


def find_time_column(records: pd.DataFrame) -> str:
    for name in ("t_ms", "time_ms", "time", "t"):
        if name in records.columns:
            return str(name)
    raise ValueError(f"records.csv has no recognizable time column. Columns: {list(records.columns)[:12]}")


def trace_column(records: pd.DataFrame, neuron_id: int) -> str | None:
    preferred = f"{int(neuron_id)}_soma_v"
    if preferred in records.columns:
        return preferred
    prefix = f"{int(neuron_id)}"
    for column in records.columns:
        name = str(column)
        if name.startswith(prefix) and name.endswith("_soma_v"):
            return name
    return None


def recorded_neuron_ids(run_dir: str | Path) -> list[int]:
    run_dir = Path(run_dir).expanduser().resolve()
    try:
        from digifly.phase2.workbench.browser_visualizer import recorded_neuron_ids as phase2_recorded_neuron_ids
        ids = phase2_recorded_neuron_ids(run_dir)
        if ids:
            return [int(x) for x in ids]
    except BaseException:
        pass
    records_path = run_dir / "records.csv"
    if not records_path.exists():
        return []
    header = records_path.open("r", encoding="utf-8").readline().strip().split(",")
    out = []
    for column in header:
        if not column.endswith("_soma_v"):
            continue
        match = re.match(r"^(\d+)", column)
        if match:
            out.append(int(match.group(1)))
    return out


def parse_neuron_ids(text: str | Iterable[int] | None, fallback: Iterable[int] = ESCAPE_CELL_IDS) -> list[int]:
    if text is None:
        return [int(x) for x in fallback]
    if isinstance(text, str):
        vals = re.findall(r"\d+", text)
        return [int(x) for x in vals] if vals else [int(x) for x in fallback]
    return [int(x) for x in text]


def plot_voltage_traces(run_dir: str | Path, neuron_ids: str | Iterable[int] | None = None, max_traces: int = 12) -> pd.DataFrame | None:
    run_dir = Path(run_dir).expanduser().resolve()
    records_path = run_dir / "records.csv"
    if not records_path.exists():
        display(Markdown(f"No `records.csv` found at `{records_path}`."))
        return None

    records = pd.read_csv(records_path)
    time_col = find_time_column(records)
    requested_ids = parse_neuron_ids(neuron_ids, fallback=recorded_neuron_ids(run_dir) or ESCAPE_CELL_IDS)
    trace_pairs = []
    missing = []
    for nid in requested_ids:
        col = trace_column(records, nid)
        if col is None:
            missing.append(nid)
        else:
            trace_pairs.append((nid, col))

    if not trace_pairs:
        display(Markdown(f"No requested soma voltage traces were found in `{records_path}`."))
        available = recorded_neuron_ids(run_dir)
        if available:
            display(Markdown(f"Recorded neuron IDs: `{available}`"))
        return records

    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        display(Markdown(f"Matplotlib is unavailable, so showing the voltage table head. Import error: `{exc}`"))
        display(records[[time_col] + [col for _, col in trace_pairs]].head())
        return records

    plot_pairs = trace_pairs[: int(max_traces)]
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for nid, col in plot_pairs:
        label = f"{nid} {CELL_LABELS.get(nid, '')}".strip()
        ax.plot(records[time_col], records[col], linewidth=1.35, label=label)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Soma voltage (mV)")
    ax.set_title(f"Phase 2 soma voltage traces: {run_dir.name}")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", ncol=2)
    if len(trace_pairs) > len(plot_pairs):
        ax.text(0.01, 0.98, f"Showing {len(plot_pairs)} of {len(trace_pairs)} traces", transform=ax.transAxes, va="top", ha="left", fontsize=9)
    fig.tight_layout()
    plt.show()

    if missing:
        display(Markdown(f"Missing requested voltage traces: `{missing}`"))
    return records


def plot_beam_waveform(df: pd.DataFrame, condition: str) -> None:
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        display(Markdown(f"Matplotlib is unavailable, so showing the first rows instead. Import error: `{exc}`"))
        display(df.head())
        return

    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(df["t_ms"], df["vertical"], label="vertical", linewidth=1.4)
    axes[0].plot(df["t_ms"], df["horizontal"], label="horizontal", linewidth=1.1, alpha=0.85)
    axes[0].set_ylabel("Beam axis signal")
    axes[0].legend(loc="best")
    axes[0].grid(alpha=0.25)

    axes[1].plot(df["t_ms"], df["vector"], label="vector", linewidth=1.4)
    axes[1].set_xlabel("Time (ms)")
    axes[1].set_ylabel("Beam vector")
    axes[1].legend(loc="best")
    axes[1].grid(alpha=0.25)

    fig.suptitle(f"Beam waveform: {condition}")
    fig.tight_layout()
    plt.show()

## Beam Waveform Launcher

Choose a paper condition and generate the flexible-beam waveform. If you paste a Phase 2 run folder with `spike_times.csv`, jump-like conditions will use the TTMn spike times from that run. If the same run has `records.csv`, the cell voltage traces will be plotted too.

In [ ]:
def run_waveform(
    condition: str = "wildtype_jump",
    output_root: str | Path = DEFAULT_OUTPUT_ROOT,
    phase2_run: str | Path | None = None,
    voltage_ids: str | Iterable[int] | None = ESCAPE_CELL_IDS,
    duration_ms: float | None = None,
    dt_ms: float | None = None,
    stim_time_ms: float = 20.0,
    seed: int = 7,
    show_beam_plot: bool = True,
    show_voltage_plot: bool = True,
) -> pd.DataFrame:
    condition = str(condition).strip()
    out_root = Path(output_root).expanduser().resolve()
    out_dir = out_root / condition
    phase2_path = None
    if phase2_run not in (None, ""):
        phase2_path = Path(phase2_run).expanduser().resolve()
        if not (phase2_path / "spike_times.csv").exists():
            display(Markdown(f"No `spike_times.csv` found in `{phase2_path}`. The beam trace will use the condition preset."))

    params = BeamParams(
        duration_ms=float(duration_ms if duration_ms is not None else default_duration_ms(condition)),
        dt_ms=float(dt_ms if dt_ms is not None else default_dt_ms(condition)),
        stim_time_ms=float(stim_time_ms),
        seed=int(seed),
    )
    df = generate_condition(condition, params, phase2_run=phase2_path if phase2_path and (phase2_path / "spike_times.csv").exists() else None)
    write_outputs(df, out_dir, condition, params, phase2_path if phase2_path and (phase2_path / "spike_times.csv").exists() else None)

    summary_path = out_dir / f"{condition}_summary.json"
    summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
    display(Markdown(f"**Wrote beam outputs to:** `{out_dir}`"))
    display(pd.Series({
        "condition": condition,
        "rows": int(df.shape[0]),
        "t_stop_ms": float(df["t_ms"].max()),
        "vertical_peak_abs": float(df["vertical"].abs().max()),
        "horizontal_peak_abs": float(df["horizontal"].abs().max()),
        "vector_peak": float(df["vector"].max()),
        "response_events": summary.get("response_events"),
    }).to_frame("value"))

    if show_beam_plot:
        plot_beam_waveform(df, condition)
    if show_voltage_plot and phase2_path is not None and (phase2_path / "records.csv").exists():
        plot_voltage_traces(phase2_path, neuron_ids=voltage_ids)
    return df


try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    condition_widget = widgets.Dropdown(options=CONDITIONS, value="wildtype_jump", description="Condition")
    output_widget = widgets.Text(value=str(DEFAULT_OUTPUT_ROOT), description="Output root", layout=widgets.Layout(width="95%"))
    phase2_widget = widgets.Text(value="", description="Phase 2 run", layout=widgets.Layout(width="95%"))
    voltage_ids_widget = widgets.Text(value=", ".join(str(x) for x in ESCAPE_CELL_IDS), description="Voltage IDs", layout=widgets.Layout(width="95%"))
    duration_widget = widgets.FloatText(value=0.0, description="Duration ms")
    dt_widget = widgets.FloatText(value=0.0, description="dt ms")
    stim_widget = widgets.FloatText(value=20.0, description="Stim ms")
    seed_widget = widgets.IntText(value=7, description="Seed")
    button = widgets.Button(description="Generate beam + voltages", button_style="primary")
    output_area = widgets.Output()

    def on_generate(_):
        with output_area:
            clear_output(wait=True)
            duration = None if duration_widget.value <= 0 else duration_widget.value
            dt = None if dt_widget.value <= 0 else dt_widget.value
            run_waveform(
                condition=condition_widget.value,
                output_root=output_widget.value,
                phase2_run=phase2_widget.value.strip() or None,
                voltage_ids=voltage_ids_widget.value,
                duration_ms=duration,
                dt_ms=dt,
                stim_time_ms=stim_widget.value,
                seed=seed_widget.value,
            )

    button.on_click(on_generate)
    display(widgets.VBox([
        condition_widget,
        output_widget,
        phase2_widget,
        voltage_ids_widget,
        widgets.HBox([duration_widget, dt_widget, stim_widget, seed_widget]),
        button,
        output_area,
    ]))
except Exception as exc:
    display(Markdown(f"Interactive widgets are unavailable: `{exc}`. Edit the values below and run the cell."))
    df = run_waveform(condition="wildtype_jump", output_root=DEFAULT_OUTPUT_ROOT, phase2_run=None)

## Plot Voltages From An Existing Phase 2 Run

Use this when you already have a Digifly run folder. Paste the folder that contains `records.csv`; the plot will use the same voltage-column conventions as the Phase 2 browser visualizer.

In [ ]:
PHASE2_RUN_DIR = ""  # Example: "/Users/juanlopez2016/Desktop/Digifly Public/Phase 1/manc_v1.2.1/export_swc/hemi_runs/single_neuron_debug"
VOLTAGE_NEURON_IDS = "10000, 10002, 10068, 10110"

if PHASE2_RUN_DIR:
    records_df = plot_voltage_traces(PHASE2_RUN_DIR, neuron_ids=VOLTAGE_NEURON_IDS)
else:
    display(Markdown("Paste a Phase 2 run folder into `PHASE2_RUN_DIR` and rerun this cell to plot soma voltages."))

## Optional: Launch The Digifly Escape Circuit Run

This cell runs the Digifly Phase 2 escape-circuit template from the notebook, then plots soma voltages and generates the matching beam waveform. Leave `RUN_DIGIFLY_ESCAPE = False` until you are ready to run NEURON locally.

In [ ]:
def _resolve_digifly_path(value: str | Path | None) -> str | None:
    if value in (None, ""):
        return None
    p = Path(value)
    if not p.is_absolute() and DIGIFLY_ROOT is not None:
        p = DIGIFLY_ROOT / p
    return str(p.expanduser().resolve())


def build_escape_config(
    run_id: str = "elliott_sparrow_2012_escape_wildtype",
    tstop_ms: float = 80.0,
    dt_ms: float = 0.025,
    iclamp_amp_nA: float = 1.0,
    iclamp_delay_ms: float = 20.0,
    iclamp_dur_ms: float = 1.0,
) -> dict:
    template_path = APP_ROOT / "configs" / "digifly_escape_run_template.json"
    cfg = json.loads(template_path.read_text(encoding="utf-8"))
    cfg["run_id"] = str(run_id)
    cfg["tstop_ms"] = float(tstop_ms)
    cfg["dt_ms"] = float(dt_ms)
    cfg["iclamp_amp_nA"] = float(iclamp_amp_nA)
    cfg["iclamp_delay_ms"] = float(iclamp_delay_ms)
    cfg["iclamp_dur_ms"] = float(iclamp_dur_ms)
    cfg["record"] = {"soma_v": "all", "spikes": "all", "spike_thresh_mV": 0.0}

    for key in ("swc_dir", "morph_swc_dir", "runs_root", "edges_root", "master_csv"):
        cfg[key] = _resolve_digifly_path(cfg.get(key))

    return cfg


def launch_digifly_escape_and_plot(
    condition: str = "wildtype_jump",
    run_id: str = "elliott_sparrow_2012_escape_wildtype",
    voltage_ids: str | Iterable[int] = "10000, 10002, 10068, 10110",
    tstop_ms: float = 80.0,
    dt_ms: float = 0.025,
    iclamp_amp_nA: float = 1.0,
    iclamp_delay_ms: float = 20.0,
    iclamp_dur_ms: float = 1.0,
):
    if DIGIFLY_ROOT is None:
        raise RuntimeError("Cannot resolve the Digifly repo root from this notebook location.")
    try:
        from digifly.phase2.api import run_walking_simulation
    except BaseException as exc:
        raise RuntimeError(f"Digifly Phase 2 API is unavailable in this kernel: {exc}") from exc

    cfg = build_escape_config(
        run_id=run_id,
        tstop_ms=tstop_ms,
        dt_ms=dt_ms,
        iclamp_amp_nA=iclamp_amp_nA,
        iclamp_delay_ms=iclamp_delay_ms,
        iclamp_dur_ms=iclamp_dur_ms,
    )
    display(Markdown(f"Launching Digifly Phase 2 run `{run_id}`..."))
    out_dir = Path(run_walking_simulation(cfg)).expanduser().resolve()
    display(Markdown(f"**Digifly run written to:** `{out_dir}`"))

    plot_voltage_traces(out_dir, neuron_ids=voltage_ids)
    beam_df = run_waveform(
        condition=condition,
        output_root=DEFAULT_OUTPUT_ROOT,
        phase2_run=out_dir,
        voltage_ids=voltage_ids,
        duration_ms=tstop_ms,
        dt_ms=dt_ms,
        stim_time_ms=iclamp_delay_ms,
        show_voltage_plot=False,
    )
    return out_dir, beam_df


RUN_DIGIFLY_ESCAPE = False

if RUN_DIGIFLY_ESCAPE:
    RUN_DIR, BEAM_DF = launch_digifly_escape_and_plot(
        condition="wildtype_jump",
        run_id="elliott_sparrow_2012_escape_wildtype",
        voltage_ids="10000, 10002, 10068, 10110",
        tstop_ms=80.0,
        dt_ms=0.025,
        iclamp_amp_nA=1.0,
        iclamp_delay_ms=20.0,
        iclamp_dur_ms=1.0,
    )
else:
    display(Markdown("Set `RUN_DIGIFLY_ESCAPE = True` and rerun this cell to launch the Digifly escape-circuit simulation."))

## Batch Generate All Paper Conditions

This produces one output folder per paper condition. It does not run NEURON; it only generates the paper proxy beam traces.

In [ ]:
RUN_BATCH = False

if RUN_BATCH:
    batch_frames = {}
    for condition in CONDITIONS:
        batch_frames[condition] = run_waveform(
            condition=condition,
            output_root=DEFAULT_OUTPUT_ROOT,
            show_beam_plot=False,
            show_voltage_plot=False,
        )
    display(Markdown(f"Generated {len(batch_frames)} conditions under `{DEFAULT_OUTPUT_ROOT}`."))
else:
    display(Markdown("Set `RUN_BATCH = True` and rerun this cell to generate every beam condition."))